# 🌿 AuraBrite Consumer Brands — Multi-Agent QnA Demo

A guided walkthrough of the enterprise QnA prototype. This notebook is designed to be pre-run and committed so reviewers see the outputs without executing anything.

**What you'll see**

1. Bootstrap the synthetic warehouse + narrative documents (deterministic seed).
2. Build the hybrid BM25 + TF-IDF RAG index.
3. Wire up the multi-agent orchestrator (LangGraph-equivalent semantics, mock LLM).
4. Ask three cross-modal enterprise questions.
5. Run the offline evaluation suite.

Everything runs offline — no API keys, no internet.

## 1. Setup

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    sys.path.insert(0, str(ROOT.parent / 'src'))

from aurabrite_qna.config import SETTINGS
from aurabrite_qna.data import connect
from aurabrite_qna.data.generator import generate_all

with connect(SETTINGS.warehouse_path) as wh:
    report = generate_all(wh, SETTINGS.docs_dir, seed=20260921)

print(report.as_text())

Warehouse tables:
  • dim_product               10 rows
  • dim_geography              6 rows
  • dim_channel                4 rows
  • dim_warehouse              6 rows
  • dim_kpi_metadata           6 rows
  • fact_sales             7,200 rows
  • fact_inventory         1,800 rows
Documents:
  • AuraGlow_Brand_Strategy_2025_2026.md
  • Supply_Chain_Disruption_Report_Q4_2025.md
  • Retail_Partnership_Notes_Tesco_Walmart_Amazon.md
  • Consumer_Insights_Q1_2026.md
  • KPI_Glossary.md


## 2. Peek at the warehouse

In [2]:
wh = connect(SETTINGS.warehouse_path)
wh.query('''
SELECT p.brand_name AS brand,
       g.region AS region,
       ROUND(SUM(s.net_revenue_usd), 0) AS nsv_usd
FROM fact_sales s
JOIN dim_product   p USING (sku_id)
JOIN dim_geography g USING (market_id)
WHERE s.date BETWEEN DATE '2025-10-01' AND DATE '2025-12-01'
GROUP BY 1, 2
ORDER BY 1, 2;
''').to_records()[:12]

[{'brand': 'AuraGlow', 'region': 'APAC', 'nsv_usd': 1674542.0},
 {'brand': 'AuraGlow', 'region': 'EMEA', 'nsv_usd': 2117539.0},
 {'brand': 'AuraGlow', 'region': 'LATAM', 'nsv_usd': 692619.0},
 {'brand': 'AuraGlow', 'region': 'North America', 'nsv_usd': 1429411.0},
 {'brand': 'CleanX', 'region': 'APAC', 'nsv_usd': 1220710.0},
 {'brand': 'CleanX', 'region': 'EMEA', 'nsv_usd': 1470561.0},
 {'brand': 'CleanX', 'region': 'LATAM', 'nsv_usd': 479297.0},
 {'brand': 'CleanX', 'region': 'North America', 'nsv_usd': 989811.0},
 {'brand': 'Dentafresh', 'region': 'APAC', 'nsv_usd': 701252.0},
 {'brand': 'Dentafresh', 'region': 'EMEA', 'nsv_usd': 866464.0},
 {'brand': 'Dentafresh', 'region': 'LATAM', 'nsv_usd': 293047.0},
 {'brand': 'Dentafresh', 'region': 'North America', 'nsv_usd': 603774.0}]

## 3. Hybrid RAG index (BM25 + TF-IDF + heading boost)

In [3]:
from aurabrite_qna.rag import HybridRetriever

retriever = HybridRetriever.from_docs_dir(SETTINGS.docs_dir, SETTINGS.vector_dir)
for hit in retriever.search('port of singapore strike apac q4 2025', top_k=3):
    print(f'[{hit.score:.4f}]  {hit.chunk.citation}')
    print(hit.chunk.text[:220], '...')
    print()

[0.0523]  Supply_Chain_Disruption_Report_Q4_2025.md § Impact on APAC (AuraGlow & Dentafresh)
Sales in APAC for **AuraGlow** and **Dentafresh** contracted by approximately
**14%** during the strike window versus the pre-strike baseline. The Mumbai and
Singapore DCs recorded a combined **32 out-of-stock events** a ...

[0.0478]  AuraGlow_Brand_Strategy_2025_2026.md § AuraGlow — Brand Strategy 2025 → 2026
**Owner:** Global Skincare CMO Office · **Classification:** Internal · Confidential ...

[0.0435]  Supply_Chain_Disruption_Report_Q4_2025.md § Global Supply-Chain Disruption Report — Q4 2025
**Prepared by:** AuraBrite Global Supply Chain COE · **Date:** 15 December 2025
**Distribution:** Executive Committee, Regional Presidents, Category GMs ...



## 4. Ask a cross-modal question end-to-end

Question: **"Why did AuraGlow sales drop in APAC in Q4 2025, and by how much?"**

Expected trace:

1. Supervisor plans → `[sql, docs]`.
2. SQL analyst pulls Q4 2025 APAC AuraGlow revenue.
3. Document researcher retrieves the supply-chain report + brand-strategy sentiment note.
4. Validator confirms coverage; synthesizer emits the final grounded answer.

In [4]:
from aurabrite_qna.agents import build_orchestrator
from IPython.display import Markdown, display

orc = build_orchestrator()
state = orc.answer('Why did AuraGlow sales drop in APAC in Q4 2025, and by how much?')

display(Markdown(state.answer))

### Answer

**Question:** Why did AuraGlow sales drop in APAC in Q4 2025, and by how much?

**Structured findings (SQL warehouse):**

| brand | net_revenue |
| --- | --- |
| AuraGlow | 1,674,542.49 |
  _SQL:_ `SELECT p.brand_name AS brand, SUM(net_revenue_usd) AS net_revenue FROM fact_sales s JOIN dim_product p USING (sku_id) JOIN dim_geography g USING (market_id) JOIN dim_channel c USING (channel_id) WHERE`

**Narrative evidence (enterprise documents):**

> **AuraGlow_Brand_Strategy_2025_2026.md § AuraGlow — Brand Strategy 2025 → 2026**
>
> **Owner:** Global Skincare CMO Office · **Classification:** Internal · Confidential

> **Supply_Chain_Disruption_Report_Q4_2025.md § Impact on APAC (AuraGlow & Dentafresh)**
>
> Sales in APAC for **AuraGlow** and **Dentafresh** contracted by approximately
> **14%** during the strike window versus the pre-strike baseline. The Mumbai and
> Singapore DCs recorded a combined **32 out-of-stock events** across the affected
> SKUs during the period. Trade partners in India (Reliance Smart, DMart) and
> Singapore (FairPrice) escalated stock complaints in weeks 43–46.

> **Supply_Chain_Disruption_Report_Q4_2025.md § Global Supply-Chain Disruption Report — Q4 2025**
>
> **Prepared by:** AuraBrite Global Supply Chain COE · **Date:** 15 December 2025
> **Distribution:** Executive Committee, Regional Presidents, Category GMs

- **Consumer_Insights_Q1_2026.md § 1. Headline Findings** — - **Clean-label demand accelerating in Health Foods.** 68% of surveyed   consumers in NA and 61% in EMEA now list "clean ingredient list" as a   top-3 purchase driver for cereals and plant proteins — up 12 pts vs 2024. -...

- **AuraGlow_Brand_Strategy_2025_2026.md § 4. Brand Sentiment Snapshot** — Net Brand Sentiment (measured on the AuraBrite Social Listening panel):  - 2024 average: **+38 pts** - 2025 Q3: **+44 pts** (peak; niacinamide launch) - 2025 Q4: **+31 pts** — *dip driven by APAC stock-outs* (see Supply-...

- **KPI_Glossary.md § AuraBrite KPI Glossary** — | KPI | Aliases | Definition | Formula | Unit | | --- | ------- | ---------- | ------- | ---- | | **Gross Revenue** (KPI-GR) | revenue|sales|top line|gross sales | Total invoiced revenue before discounts, returns, and re...


**Confidence:** 0.72 · validator: pass (1 round(s))

In [5]:
import pandas as pd

trace_df = pd.DataFrame([
    {'node': t.node, 'latency_ms': round(t.latency_ms, 2), 'summary': t.output_summary}
    for t in state.trace
])
trace_df

,node,latency_ms,summary
0,supervisor,0.11,"plan=['sql', 'docs']"
1,sql_analyst,4.81,"1 rows · brand, net_revenue"
2,doc_researcher,0.66,6 chunks
3,validator,0.07,pass (conf=0.72)
4,synthesizer,0.02,2284 chars


## 5. A numeric + derived-analytics question

*"How much did NutriVita e-commerce revenue grow between H1 2025 and H1 2026, and why?"*

Expected trace: `sql → docs → python → validator → synth`.

In [6]:
state2 = orc.answer('How much did NutriVita e-commerce revenue grow in H1 2026, and why?')
display(Markdown(state2.answer))

### Answer

**Question:** How much did NutriVita e-commerce revenue grow in H1 2026, and why?

**Structured findings (SQL warehouse):**

| brand | net_revenue |
| --- | --- |
| NutriVita | 1,695,684.04 |
  _SQL:_ `SELECT p.brand_name AS brand, SUM(net_revenue_usd) AS net_revenue FROM fact_sales s JOIN dim_product p USING (sku_id) JOIN dim_geography g USING (market_id) JOIN dim_channel c USING (channel_id) WHERE`

**Narrative evidence (enterprise documents):**

> **AuraGlow_Brand_Strategy_2025_2026.md § AuraGlow — Brand Strategy 2025 → 2026**
>
> **Owner:** Global Skincare CMO Office · **Classification:** Internal · Confidential

> **Consumer_Insights_Q1_2026.md § Consumer Insights — Q1 2026**
>
> **Prepared by:** AuraBrite Consumer & Market Insights · **Date:** April 2026

> **Retail_Partnership_Notes_Tesco_Walmart_Amazon.md § 3. Amazon (Global E-Commerce)**
>
> - Contract is a global co-op agreement, not per country.
> - **E-Commerce rebate & funded-promo bundle: 18%** (matches internal EC discount
>   guardrail set by CMO office).
> - Category performance FY25: NutriVita +14% (subscribe-and-save), AuraGlow +11%,
>   Dentafresh flat.
> - JBP asks 2026: Prime Day exclusive AuraGlow bundle; investment in the
>   *NutriVita Plant Protein 500g* subscription ladder.

- **AuraGlow_Brand_Strategy_2025_2026.md § 6. Risks** — 1. Continued niacinamide shortage (see supply report) — could erode APAC targets. 2. Competitive premiumisation in North America. 3. Discount creep in E-Commerce past the 18% guardrail.

- **KPI_Glossary.md § AuraBrite KPI Glossary** — | KPI | Aliases | Definition | Formula | Unit | | --- | ------- | ---------- | ------- | ---- | | **Gross Revenue** (KPI-GR) | revenue|sales|top line|gross sales | Total invoiced revenue before discounts, returns, and re...

- **Consumer_Insights_Q1_2026.md § 1. Headline Findings** — - **Clean-label demand accelerating in Health Foods.** 68% of surveyed   consumers in NA and 61% in EMEA now list "clean ingredient list" as a   top-3 purchase driver for cereals and plant proteins — up 12 pts vs 2024. -...


**Confidence:** 0.72 · validator: pass (1 round(s))

## 6. Offline evaluation

In [7]:
from aurabrite_qna.eval import run_evaluation

report = run_evaluation()
pd.DataFrame([report.summary])

,n_questions,n_passed,pass_rate,avg_keyword_recall,avg_citation_recall,avg_routing_precision,avg_routing_recall,avg_latency_ms
0,8,8,1.0,0.875,1.0,0.938,1.0,3.2


In [8]:
pd.DataFrame([
    {
        'id': i.id,
        'passed': i.passed,
        'keyword_recall': i.keyword_recall,
        'citation_recall': i.citation_recall,
        'routing_recall': i.routing_recall,
        'latency_ms': i.latency_ms,
    }
    for i in report.items
])

,id,passed,keyword_recall,citation_recall,routing_recall,latency_ms
0,q1-apac-auraglow-q4,True,1.0,1.0,1.0,4.3
1,q2-nutrivita-ec-lift,True,1.0,1.0,1.0,3.9
2,q3-dentafresh-emea-decline,True,0.5,1.0,1.0,4.7
3,q4-top-brand-modern-trade-2025,True,1.0,1.0,1.0,3.4
4,q5-tesco-rebate,True,1.0,1.0,1.0,0.7
5,q6-kpi-definition,True,1.0,1.0,1.0,0.6
6,q7-auraglow-campaign-spend,True,0.5,1.0,1.0,4.4
7,q8-yoy-nutrivita,True,1.0,1.0,1.0,4.0


## 7. Architecture at a glance

```text
           ┌───────────────────┐
  user →   │   Supervisor      │  (plans which workers to invoke)
           └────────┬──────────┘
                    │ plan = [sql?, docs?, web?, python?]
     ┌─────────┬────┴────┬─────────┬──────────┐
     ▼         ▼         ▼         ▼          
  ┌─────┐  ┌──────┐  ┌──────┐  ┌──────────┐   
  │ SQL │  │ Docs │  │ Web  │  │ Python   │   
  │DuckD│  │BM25+ │  │DDG /  │  │AST-safe  │   
  │B/SQL│  │TF-IDF│  │Tavily │  │sandbox   │   
  └──┬──┘  └───┬──┘  └───┬──┘  └────┬─────┘   
     └─────────┴─────────┴──────────┘          
                    │ Evidence[]                
                    ▼                           
           ┌───────────────────┐                
           │   Validator       │ ── revise? ──┐  
           └────────┬──────────┘              │  
                    │ pass                    │  
                    ▼                         │  
           ┌───────────────────┐              │  
           │  Synthesizer      │              │  
           └────────┬──────────┘              │  
                    ▼                         │  
               final answer                   │  
                                              │  
         ▲───────────────── loop (bounded) ───┘  
```

The graph is implemented once in `aurabrite_qna.agents.orchestrator` and mirrored 1-to-1 in `langgraph_runner.py` so you can drive it through a real `langgraph.StateGraph` when LangGraph is installed.